In [1]:
import pandas as pd

train_df = pd.read_csv("../datasets/processed/train.csv") 

lengths = train_df["Query"].str.len()

print("Percentile 90:", lengths.quantile(0.90))
print("Percentile 95:", lengths.quantile(0.95))
print("Percentile 99:", lengths.quantile(0.99))
print("Max:", lengths.max())
print("Mean:", lengths.mean())

all_chars = set()
for q in train_df["Query"]:
    all_chars.update(q)

print("Số ký tự unique (vocab size ước lượng):", len(all_chars))
print("Danh sách ký tự:", sorted(all_chars))

Percentile 90: 154.0
Percentile 95: 221.0
Percentile 99: 377.0
Max: 5370
Mean: 68.91991587121825
Số ký tự unique (vocab size ước lượng): 105
Danh sách ký tự: ['\x18', ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '=', '>', '?', '@', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', '\\', ']', '^', '_', '`', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '{', '|', '~', '\x80', '\x98', '\xa0', 'â', 'ü', '‘', '’', '거', '리', '트']


In [2]:
from collections import Counter

char_counts = Counter()
for q in train_df["Query"]:
    char_counts.update(q)

for ch, cnt in sorted(char_counts.items(), key=lambda x: x[1]):
    print(repr(ch), cnt)

'\x18' 1
'â' 2
'\x80' 2
'\x98' 2
'트' 2
'리' 2
'거' 2
'’' 3
'‘' 3
'ü' 4
'^' 5
'{' 22
'&' 27
'`' 36
'\xa0' 48
'!' 57
'Z' 65
'X' 70
'~' 83
'Q' 86
'\\' 111
'?' 186
'<' 187
'>' 189
'[' 330
']' 330
'K' 488
':' 564
';' 569
'G' 901
'J' 909
'V' 982
'$' 1003
'/' 1021
'@' 1052
'j' 1555
'#' 1592
'B' 1763
'Y' 1775
'z' 1895
'q' 1950
'P' 1971
'%' 2260
'U' 2398
'+' 3523
'A' 4071
'H' 4495
'D' 4572
'v' 4607
'k' 4710
'W' 5063
'x' 5475
'.' 5561
'_' 6070
'"' 6326
'I' 7049
'*' 7900
'g' 8671
'w' 9179
'N' 9367
'b' 9390
'-' 9512
'f' 9644
'y' 10800
'M' 10996
'F' 11607
'9' 12741
'8' 13291
'L' 13642
'p' 13729
'C' 14161
'=' 14237
'4' 14473
'|' 14599
'S' 15265
'6' 15394
'm' 16050
'2' 17048
'5' 17351
'O' 17832
'T' 18206
'3' 18397
'R' 18427
'7' 18992
"'" 20605
'u' 21111
',' 21703
'd' 22076
'h' 24143
'i' 24924
'0' 25696
'o' 33511
'c' 35428
'(' 36778
'E' 36975
'n' 38604
't' 39502
'1' 40045
')' 40777
'l' 44109
'a' 49163
'r' 49217
's' 50678
'e' 77445
' ' 502560


In [3]:
# Bước 1: xem quy mô ảnh hưởng — có bao nhiêu dòng thực sự bị cắt ở max_len=250
long_queries = train_df[train_df["Query"].str.len() > 250]
print("Số dòng dài hơn 250:", len(long_queries))
print(long_queries["Label"].value_counts())

# Bước 2: với các câu SQLi (label=1) dài hơn 250, kiểm tra các từ khóa/ký tự
# nguy hiểm nằm ở vị trí nào trong câu (0 = đầu câu, 1 = cuối câu)
def keyword_position_ratio(query, keyword):
    idx = query.upper().find(keyword.upper())
    if idx == -1:
        return None
    return idx / len(query)

long_sqli = long_queries[long_queries["Label"] == 1]["Query"]

for keyword in ["UNION", "--", "#", "/*", " OR "]:
    ratios = long_sqli.apply(lambda q: keyword_position_ratio(q, keyword)).dropna()
    if len(ratios) > 0:
        print(f"{keyword!r}: số dòng chứa = {len(ratios)}, vị trí trung bình = {ratios.mean():.2f} (0=đầu, 1=cuối)")
    else:
        print(f"{keyword!r}: không có dòng nào chứa từ khóa này trong nhóm dài >250")

Số dòng dài hơn 250: 1082
Label
1    1046
0      36
Name: count, dtype: int64
'UNION': số dòng chứa = 86, vị trí trung bình = 0.71 (0=đầu, 1=cuối)
'--': số dòng chứa = 268, vị trí trung bình = 0.99 (0=đầu, 1=cuối)
'#': số dòng chứa = 25, vị trí trung bình = 0.99 (0=đầu, 1=cuối)
'/*': không có dòng nào chứa từ khóa này trong nhóm dài >250
' OR ': số dòng chứa = 464, vị trí trung bình = 0.06 (0=đầu, 1=cuối)


In [6]:
import json

# Bước 1: đếm tần suất ký tự (giống bước đã làm trước đó)
from collections import Counter
char_counts = Counter()
for q in train_df["Query"]:
    char_counts.update(q)

# Bước 2: lọc theo ngưỡng tần suất >= 10, sắp xếp để thứ tự ổn định (không đổi mỗi lần chạy lại)
kept_chars = sorted([ch for ch, cnt in char_counts.items() if cnt >= 10])

print("Số ký tự giữ lại:", len(kept_chars))  # kỳ vọng: 94

# Bước 3: xây mapping ký tự -> index
# <PAD>=0, <UNK>=1, ký tự thật bắt đầu từ index 2
vocab = {"<PAD>": 0, "<UNK>": 1}
for i, ch in enumerate(kept_chars):
    vocab[ch] = i + 2

print("Vocab size:", len(vocab))  # kỳ vọng: 96

# Bước 4: lưu vocab.json
with open("../artifacts/vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)

# Bước 5: lưu max_len.json — gồm cả max_len và ghi chú chiến lược truncating
# để Tuần 5 không cần quay lại đọc báo cáo mới nhớ ra cách xử lý
max_len_config = {
    "max_len": 250,
    "truncating_strategy": "head_tail",
    "head_len": 125,
    "tail_len": 125,
    "padding": "post",
    "pad_token_index": 0
}

with open("../artifacts/max_len.json", "w", encoding="utf-8") as f:
    json.dump(max_len_config, f, ensure_ascii=False, indent=2)

print("Đã lưu artifacts/vocab.json và artifacts/max_len.json")

Số ký tự giữ lại: 94
Vocab size: 96
Đã lưu artifacts/vocab.json và artifacts/max_len.json


In [8]:
with open("../artifacts/vocab.json", encoding="utf-8") as f:
    check = json.load(f)
print(len(check), check)

96 {'<PAD>': 0, '<UNK>': 1, ' ': 2, '!': 3, '"': 4, '#': 5, '$': 6, '%': 7, '&': 8, "'": 9, '(': 10, ')': 11, '*': 12, '+': 13, ',': 14, '-': 15, '.': 16, '/': 17, '0': 18, '1': 19, '2': 20, '3': 21, '4': 22, '5': 23, '6': 24, '7': 25, '8': 26, '9': 27, ':': 28, ';': 29, '<': 30, '=': 31, '>': 32, '?': 33, '@': 34, 'A': 35, 'B': 36, 'C': 37, 'D': 38, 'E': 39, 'F': 40, 'G': 41, 'H': 42, 'I': 43, 'J': 44, 'K': 45, 'L': 46, 'M': 47, 'N': 48, 'O': 49, 'P': 50, 'Q': 51, 'R': 52, 'S': 53, 'T': 54, 'U': 55, 'V': 56, 'W': 57, 'X': 58, 'Y': 59, 'Z': 60, '[': 61, '\\': 62, ']': 63, '_': 64, '`': 65, 'a': 66, 'b': 67, 'c': 68, 'd': 69, 'e': 70, 'f': 71, 'g': 72, 'h': 73, 'i': 74, 'j': 75, 'k': 76, 'l': 77, 'm': 78, 'n': 79, 'o': 80, 'p': 81, 'q': 82, 'r': 83, 's': 84, 't': 85, 'u': 86, 'v': 87, 'w': 88, 'x': 89, 'y': 90, 'z': 91, '{': 92, '|': 93, '~': 94, '\xa0': 95}


In [9]:
from sklearn.model_selection import train_test_split
import os

os.makedirs("../datasets/processed", exist_ok=True)

train_split, val_split = train_test_split(
    train_df,
    test_size=0.20,
    stratify=train_df["Label"],
    random_state=42
)

print("train_split:", len(train_split), "dòng")
print("val_split:", len(val_split), "dòng")
print("\nTỷ lệ nhãn train_split:")
print(train_split["Label"].value_counts(normalize=True))
print("\nTỷ lệ nhãn val_split:")
print(val_split["Label"].value_counts(normalize=True))

train_split.to_csv("../datasets/processed/train_split.csv", index=False)
val_split.to_csv("../datasets/processed/val_split.csv", index=False)

print("\nĐã lưu train_split.csv và val_split.csv")

train_split: 19779 dòng
val_split: 4945 dòng

Tỷ lệ nhãn train_split:
Label
0    0.631832
1    0.368168
Name: proportion, dtype: float64

Tỷ lệ nhãn val_split:
Label
0    0.631951
1    0.368049
Name: proportion, dtype: float64

Đã lưu train_split.csv và val_split.csv
